In [2]:
!python --version

Python 3.11.13


In [3]:
!pip -q uninstall -y deepctr tensorflow tensorflow-cpu tensorflow-rocm tf-keras keras keras-core keras-nlp \
  jax jaxlib tensorstore spacy thinc ml-dtypes numpy h5py
!pip -q install --upgrade pip setuptools wheel


In [4]:
!pip -q uninstall -y chex distrax flax optax orbax-checkpoint jax jaxlib tensorstore


In [5]:
!pip -q install "numpy==1.26.4" "ml-dtypes==0.2.0" "h5py==3.10.0"
!pip -q install "tensorflow==2.15.0.post1" "keras==2.15.0"
!pip -q install --no-deps "deepctr==0.9.3"   # 의존성은 우리가 고정했으니 no-deps 권장


In [6]:
import tensorflow as tf, deepctr
print("TF:", tf.__version__)            # 2.15.0.post1
print("DeepCTR:", deepctr.__version__)  # 0.9.3


TF: 2.15.0
DeepCTR: 0.9.3


In [7]:
# --- DeepCTR 임포트 이전 셀 ---
import os, sys, types
os.environ["TF_USE_LEGACY_KERAS"] = "1"   # 안전하게 Keras2 경로 유지

import tensorflow as tf
print("TF:", tf.__version__)  # 2.15.x 이어야 함

# 핵심: tensorflow.keras 대신 'keras' 패키지에서 직접 가져오기
from keras.layers import LSTM, Lambda, Dropout, Layer

# DeepCTR가 참조하는 내부 경로(tensorflow.python.keras.layers)에 심볼 주입
shim = types.ModuleType("tensorflow.python.keras.layers")
shim.LSTM = LSTM
shim.Lambda = Lambda
shim.Layer = Layer
shim.Dropout = Dropout
sys.modules["tensorflow.python.keras.layers"] = shim


TF: 2.15.0


In [8]:
import pkgutil, importlib, importlib.metadata as md
print("tensorflow==", md.version("tensorflow"))
print("keras==", md.version("keras"))
print("deepctr==", md.version("deepctr"))


tensorflow== 2.15.0.post1
keras== 2.15.0
deepctr== 0.9.3


In [9]:
# === DeepCTR 임포트 이전 셀 ===
import os, sys, types
os.environ["TF_USE_LEGACY_KERAS"] = "1"   # 안전: Keras2 경로 유지

import tensorflow as tf
import keras

# 1) 내부 경로 -> keras.layers 전체를 프록시로 매핑
sys.modules["tensorflow.python.keras.layers"] = keras.layers

# 2) 내부 경로 -> keras.initializers 전체를 프록시로 매핑
sys.modules["tensorflow.python.keras.initializers"] = keras.initializers

# 3) init_ops_v2가 필요한 심볼들을 keras.initializers에서 노출
from keras.initializers import TruncatedNormal, Constant, glorot_uniform
init_v2 = types.ModuleType("tensorflow.python.ops.init_ops_v2")
init_v2.TruncatedNormal = TruncatedNormal
init_v2.Constant = Constant
init_v2.glorot_uniform = glorot_uniform
sys.modules["tensorflow.python.ops.init_ops_v2"] = init_v2

print("TF:", tf.__version__, "| Keras:", keras.__version__, "→ shim installed")


TF: 2.15.0 | Keras: 2.15.0 → shim installed


In [10]:
from deepctr.feature_column import SparseFeat, DenseFeat, VarLenSparseFeat, get_feature_names
from deepctr.models import DeepFM

print("DeepCTR import OK")


DeepCTR import OK


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import roc_auc_score, log_loss, average_precision_score
from deepctr.feature_column import SparseFeat, DenseFeat, VarLenSparseFeat, get_feature_names
from deepctr.models import DeepFM
import numpy as np


In [12]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

train= pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/CRT/train_part1.parquet' , engine= 'pyarrow')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
df = train.copy()

In [14]:
df = df.drop(['l_feat_20' , 'l_feat_23','l_feat_2','l_feat_24'], axis = 1 )
df = df.drop(['history_a_1' , 'history_a_2' , 'history_a_3'], axis= 1)
df = df[~df['inventory_id'].isin([92, 21])]

    max_len : 800
    topk : 1000
    min_count = 500

In [ ]:
# df['seq'] = df['seq'].apply(lambda s: [int(t) for t in str(s).split(',') if t.strip()!=''])

# df['seq'] = df['seq'].apply(lambda lst: [x+1 for x in lst if x >= 0])

# df['seq_len_raw'] = df['seq'].apply(len)

In [18]:
import numpy as np

def parse_inc_to_array(s: str) -> np.ndarray:
    # 문자열 "1,2,3" → int32 배열 [1,2,3]
    a = np.fromstring(str(s), sep=',', dtype=np.int32)
    if a.size == 0:
        return a
    # 음수 제외하고 +1
    return a[a >= 0] + 1

def count_nonneg_len(s: str) -> int:
    a = np.fromstring(str(s), sep=',', dtype=np.int32)
    return int((a >= 0).sum())

df["seq_arr"] = df["seq"].map(parse_inc_to_array)    # 각 row를 np.int32 배열로 변환
df["seq_len_raw"] = df["seq"].map(count_nonneg_len)  # 길이만 계산


In [20]:
# padding
from tensorflow.keras.preprocessing.sequence import pad_sequences
MAX_LEN = 150

seq_padding = pad_sequences(
    df["seq_arr"].tolist(),
    maxlen = MAX_LEN,
    dtype = 'int32',
    padding = 'post',
    truncating = 'pre',
    value = 0
)

df['seq_len'] = np.minimum(df['seq_len_raw'].values, MAX_LEN).astype('int32')


### 피처 열 생성


> DeepCTR에서 SparseFeat는 내부적으로 Embedding Lookup을 하기 때문에, 입력값은 반드시 0 ~ (vocabulary_size-1) 범위의 연속 정수 인덱스여야 한다.

>
    SparseFeat("inventory_id", vocabulary_size=16) 같이 정의하면, DeepCTR은 입력값을 0~15 정수 인덱스로 간주한다.

    그런데 실제 데이터는 2.0, 36.0, 37.0, …, 95.0처럼 흩어져 있음.

    이걸 그대로 Embedding lookup에 넣으면 → 인덱스 범위 초과 오류 또는 embedding index mismatch 발생.

> label encoding으로 맞춘다.

In [36]:
label_feat = ['gender', 'age_group', 'day_of_week']

encoders = {}

for feat in label_feat:
    le = LabelEncoder()
    df[feat] = le.fit_transform(df[feat])
    encoders[feat] = le

In [37]:
HASH_BUCKET = 1_000_000
MAX_LEN = 150


sparse_fixed = [
    SparseFeat('gender', vocabulary_size= df["gender"].nunique() , embedding_dim= 4),
    SparseFeat('age_group' , vocabulary_size= df["age_group"].nunique() , embedding_dim= 8),
    SparseFeat('day_of_week', vocabulary_size=df["day_of_week"].nunique() , embedding_dim= 8),
    SparseFeat('hour', vocabulary_size=24 , embedding_dim=16),
]

sparse_hash = [
    SparseFeat('inventory_id' , vocabulary_size= HASH_BUCKET, embedding_dim= 8 , embedding_name = 'lala' , use_hash= True),
    SparseFeat('l_feat_14', vocabulary_size= HASH_BUCKET, embedding_dim= 16 , use_hash= True ),
]

ordinal_sparse = [
    SparseFeat('l_feat_3' ,vocabulary_size= 3 , embedding_dim= 4),
    SparseFeat('l_feat_27',vocabulary_size= 5 , embedding_dim= 4),
    SparseFeat('feat_e_4' ,vocabulary_size= 4 , embedding_dim= 4),
    SparseFeat('feat_a_1' ,vocabulary_size= 5 , embedding_dim= 4),
    SparseFeat('feat_a_3' ,vocabulary_size= 6 , embedding_dim= 8),
    SparseFeat('feat_a_4' ,vocabulary_size= 6 , embedding_dim= 8),
    SparseFeat('feat_a_8' ,vocabulary_size= 7 , embedding_dim= 8),
    SparseFeat('feat_a_13',vocabulary_size= 5 , embedding_dim= 4),
    SparseFeat('feat_a_16',vocabulary_size= 7 , embedding_dim= 8),
    SparseFeat('feat_a_18',vocabulary_size= 7 , embedding_dim= 8)
]

# ordinal_scores = [
#     DenseFeat('l_feat_3_ordscore', 1),
#     DenseFeat('feat_a_1_ordscore', 1),


varlen_seq  = VarLenSparseFeat(
    sparsefeat = SparseFeat('seq' ,
                            vocabulary_size= HASH_BUCKET , #2715931
                            embedding_dim= 32 ,
                            use_hash=True ,
                            embedding_name = 'lala'),
    maxlen = MAX_LEN,
    combiner = 'mean',
    length_name= 'seq_len',
    weight_name = None,
    weight_norm = False
)


seq_col = 'seq'
label_col = 'clicked'
seq_len_col = 'seq_len'

# dense feats 자동 수집
nominal_names = [f.name for f in (sparse_fixed + sparse_hash)]
ordinal_names = [f.name for f in ordinal_sparse]

exclude = set(nominal_names + ordinal_names + [label_col, seq_col, seq_len_col])
dense_feats = [DenseFeat(c, 1) for c in df.columns if c not in exclude]



In [42]:
# 연속형 인코딩
dense_feat_names = [f.name for f in dense_feats]
drop_cols  = {"seq_arr", "seq_len_raw", "seq_len"}
dense_feat_names = [c for c in dense_feat_names if c not in drop_cols]



mms = MinMaxScaler(feature_range=(0, 1))
df[dense_feat_names] = mms.fit_transform(df[dense_feat_names])


In [43]:
linear_feature_columns = sparse_fixed + sparse_hash + ordinal_sparse + dense_feats + [varlen_seq]
dnn_feature_columns    = sparse_fixed + sparse_hash + ordinal_sparse + dense_feats + [varlen_seq]
feature_names = get_feature_names(linear_feature_columns + dnn_feature_columns)

# DIN: 고정길이 + 시퀀스(VarLenSparseFeat) 함께 사용
# dnn_feature_columns_din = linear_feature_columns + [varlen_seq]
# behavior_feature_list = ['inventory_id']  # query(현재 타깃)와 history 키 그룹 매칭
